# Figure 1 — a probabilistic mouse–human coupling

This notebook rebuilds every panel of Figure 1 from the coupling and the parcel tables. Each
number is computed here and compared against the value printed in the manuscript, which is
written inline as a constant, so nothing under `outputs/logs/` is read.

Before running: `python scripts/fetch_data.py` (coupling, parcel tables, reference volumes).

| panel | what it shows | built in |
|---|---|---|
| a | homologous systems differ several-fold in relative size | §5 |
| b | some regions have no one-to-one counterpart | §5 |
| c | 21 Garin anchor classes and 26 curated packs | §5 |
| d | row-wise argmax of π | §1 |
| e | π aggregated to the 21 homology classes | §2 |
| f | the same matrix as a mouse→human flow | §3 |
| g | worked queries routed onto the human brain | §4 |

Panels a, b, c and g are volumetric renderings, so they are produced by their panel scripts
rather than re-implemented here. Those scripts are self-contained and are called in §4 and §5.

In [ ]:
import subprocess, sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib.path import Path as MPath
from matplotlib.patches import PathPatch
from scipy.stats import pearsonr

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
FIGS = ROOT.parent / 'manuscript' / 'figures'
sys.path.insert(0, str(ROOT / 'src'))
sys.path.insert(0, str(FIGS))

from otter.data import load_cached, load_pi, pi_provenance
# region_aggregate is the SHARED 21x21 collapse used by the Fig 1e panel script and the Fig 1f
# ribbon script. Re-implementing it here is exactly how this notebook once reported 0.275 where
# the panel showed 0.262, so it is imported rather than copied.
from _common import coarse_region, row_normalise, region_aggregate, GARIN_NAMES

pi = load_pi()                 # load_pi() owns which coupling is canonical -- never hardcode
M, _ = load_cached('mouse', cache_dir=str(ROOT / 'outputs/anndata'))
H, _ = load_cached('human', cache_dir=str(ROOT / 'outputs/anndata'))

prov = pi_provenance()
print(f"coupling {pi.shape[0]:,} mouse x {pi.shape[1]:,} human parcels")
print(f"  file   {prov['pi_file']}")
print(f"  sha256 {prov['pi_sha256'][:16]}...")

# Values as printed in the manuscript. The notebook recomputes each one and compares; it does
# not read them from a results file. If a check fails, the text and the code have diverged.
PUBLISHED = {
    'median top-target probability':      (0.31,  0.005),
    'fraction of parcels above 0.5':      (0.20,  0.005),
    'mean self-mass on the diagonal':     (0.40,  0.005),
    'fold enrichment over uniform':       (8.5,   0.15),
    'topographic fidelity r':             (0.53,  0.02),
}

def check(name, value):
    expect, tol = PUBLISHED[name]
    ok = abs(value - expect) <= tol
    print(f"  [{'ok ' if ok else 'FAIL'}] {name:36s} computed {value:.4f}   manuscript {expect}")
    return ok

## 1. How concentrated is the coupling? (Fig. 1d)

Row-normalise π so that each mouse parcel becomes a probability distribution over the 2,094 human
parcels, then ask how much mass its single best partner carries.

The coupling is soft, and that is a choice rather than a finding. Concentration is set by the
entropic regularisation ε: re-fitting at ε = 0.005 gives a near-deterministic coupling with no
gain in held-out homology recovery. We select ε by held-out recovery and leave the spread visible.

In [ ]:
P = row_normalise(pi)
top_p = P.max(axis=1)
med, frac_sharp = float(np.median(top_p)), float((top_p > 0.5).mean())

print(f"top-target probability: median {med:.4f}, mean {top_p.mean():.3f}")
print(f"{frac_sharp * 100:.1f} % of mouse parcels place > 0.5 of their mass on one human parcel\n")
check('median top-target probability', med)
check('fraction of parcels above 0.5', frac_sharp)

In [ ]:
# ---------------- Fig 1d ----------------
amax = P.argmax(axis=1)
fig, ax = plt.subplots(figsize=(7.0, 6.2))
sc = ax.scatter(amax, np.arange(pi.shape[0]), s=6, c=top_p, cmap='viridis',
                vmin=0, vmax=1, linewidths=0)
ax.set_xlim(0, pi.shape[1]); ax.set_ylim(pi.shape[0], 0)
ax.set_xlabel('human parcel (of 2,094)'); ax.set_ylabel('mouse parcel (of 1,864)')
fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.02).set_label('top-target probability')
ax.set_title("Each mouse parcel at its most probable human counterpart\n"
             f"median top-target probability {med:.2f};  > 0.5 for {frac_sharp * 100:.0f} % of parcels",
             fontweight='bold', loc='left', fontsize=10.5)
plt.show()

print("Parcels are indexed in anatomical order in both species, so the diagonal means routing")
print("preserves topography. Section 3 tests that rather than eyeballing it.")

## 2. Where does the mass land? (Fig. 1e)

Aggregating π to the 21 Garin homology classes and row-normalising gives, on the diagonal, the
fraction of each mouse class's mass that lands on its own human class.

`region_aggregate` is imported rather than reimplemented here; see the note in the setup cell.

In [ ]:
Crow, NAMES = region_aggregate(pi, M, H)
K = len(NAMES)

self_mass = float(np.mean(np.diag(Crow)))
# The uniform baseline is the mean share of human parcels a class occupies, not 1/21: classes
# differ in size, so 1/21 would understate what a size-matched random mapping achieves.
hc = coarse_region(H.var)
uniform = float(np.mean([(hc == k).sum() / len(hc) for k in range(1, K + 1) if (hc == k).any()]))

print(f"mean self-mass on the homologous diagonal : {self_mass:.4f}")
print(f"expected under a size-matched uniform map : {uniform:.4f}")
print(f"fold enrichment                           : {self_mass / uniform:.2f}x\n")
check('mean self-mass on the diagonal', self_mass)
check('fold enrichment over uniform', self_mass / uniform)

In [ ]:
# ---------------- Fig 1e ----------------
fig, ax = plt.subplots(figsize=(8.2, 7.0))
im = ax.imshow(Crow, cmap='magma', vmin=0, vmax=1)
ax.set_xticks(range(K)); ax.set_xticklabels(NAMES, rotation=90, fontsize=7.5)
ax.set_yticks(range(K)); ax.set_yticklabels(NAMES, fontsize=7.5)
ax.set_xlabel('human region'); ax.set_ylabel('mouse region')
for i in range(K):
    for j in range(K):
        if Crow[i, j] > 0.25:
            ax.text(j, i, f"{Crow[i, j]:.2f}", ha='center', va='center', fontsize=5.5,
                    color='white' if Crow[i, j] < 0.6 else 'black')
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02).set_label('fraction of mouse-region mass')
ax.set_title("Coupling aggregated to the 21 Garin homology classes\n"
             f"mean self-mass {self_mass:.2f} versus {uniform:.3f} uniform ({self_mass / uniform:.1f}x)",
             fontweight='bold', loc='left', fontsize=10.5)
plt.show()

d = np.diag(Crow)
order = np.argsort(d)[::-1]
print('strongest self-correspondence:', ', '.join(f"{NAMES[i]} {d[i]:.2f}" for i in order[:4]))
print('weakest self-correspondence  :', ', '.join(f"{NAMES[i]} {d[i]:.2f}" for i in order[-4:]))

## 3. The same matrix as a flow (Fig. 1f)

Panel f redraws panel e as a bipartite ribbon diagram, which makes off-diagonal routing easier to
read. It uses the same `Crow` matrix.

In [ ]:
# ---------------- Fig 1f ----------------
pal = (list(plt.get_cmap('tab20').colors) + list(plt.get_cmap('tab20b').colors))[:K]

def ribbon(ax, y0, y1, h, color, xa, xb, alpha):
    cx = (xa + xb) / 2
    verts = [(xa, y0-h), (cx, y0-h), (cx, y1-h), (xb, y1-h),
             (xb, y1+h), (cx, y1+h), (cx, y0+h), (xa, y0+h), (xa, y0-h)]
    codes = [MPath.MOVETO, MPath.CURVE4, MPath.CURVE4, MPath.CURVE4,
             MPath.LINETO, MPath.CURVE4, MPath.CURVE4, MPath.CURVE4, MPath.CLOSEPOLY]
    ax.add_patch(PathPatch(MPath(verts, codes), facecolor=color, edgecolor='none', alpha=alpha))

fig, ax = plt.subplots(figsize=(8.4, 8.6)); ax.axis('off')
ys = np.linspace(0.96, 0.04, K); xa, xb, THR, maxw = 0.30, 0.70, 0.06, 0.018
for i, j, v in sorted([(i, j, Crow[i, j]) for i in range(K) for j in range(K)
                       if Crow[i, j] > THR], key=lambda c: c[2]):
    ribbon(ax, ys[i], ys[j], maxw * v / Crow.max() + 0.001, pal[i], xa, xb, 0.8 if i == j else 0.55)
for i, (y, lab) in enumerate(zip(ys, NAMES)):
    ax.add_patch(plt.Rectangle((xa - 0.012, y - 0.016), 0.012, 0.032, color=pal[i]))
    ax.add_patch(plt.Rectangle((xb, y - 0.016), 0.012, 0.032, color=pal[i]))
    ax.text(xa - 0.02, y, lab, ha='right', va='center', fontsize=8)
    ax.text(xb + 0.02, y, lab, ha='left', va='center', fontsize=8)
ax.text(xa - 0.02, 1.0, 'MOUSE region', ha='right', va='center', fontsize=10, fontweight='bold')
ax.text(xb + 0.02, 1.0, 'HUMAN region', ha='left', va='center', fontsize=10, fontweight='bold')
ax.set_title('Region-to-region coupling  (ribbon width ∝ routed mass)',
             fontsize=12, fontweight='bold', loc='left')
ax.set_xlim(0, 1); ax.set_ylim(0, 1.03)
plt.show()

## 4. Does routing preserve topography? (Fig. 1g)

The diagonal in panel d could in principle be a solver artefact, so it is worth testing directly:
do two mouse parcels that lie close together route to human centroids that lie close together?
The null shuffles π's rows, which breaks the correspondence while preserving every marginal.

In [ ]:
mouse_xyz = M.var[['x', 'y', 'z']].to_numpy(float)
human_xyz = H.var[['x', 'y', 'z']].to_numpy(float)

def routed_centroids(p):
    return (p / np.maximum(p.sum(axis=1, keepdims=True), 1e-12)) @ human_xyz

def pdist_flat(X):
    D = np.linalg.norm(X[:, None, :] - X[None, :, :], axis=-1)
    return D[np.triu_indices(len(X), k=1)]

rng = np.random.default_rng(0)
sub = rng.choice(pi.shape[0], size=900, replace=False)     # ~400k pairs; full set is O(n^2) memory
dm = pdist_flat(mouse_xyz[sub])
r_obs = float(pearsonr(dm, pdist_flat(routed_centroids(pi)[sub]))[0])
null = np.array([float(pearsonr(dm, pdist_flat(
    routed_centroids(pi[rng.permutation(pi.shape[0])])[sub]))[0]) for _ in range(20)])

print(f"mouse pairwise distance vs routed human pairwise distance:  r = {r_obs:.3f}")
print(f"permuted-coupling null: r = {null.mean():+.3f} +/- {null.std():.3f}\n")
check('topographic fidelity r', r_obs)

In [ ]:
# Parcel-level labels exist only for the curated anchors, so the meaningful query is at the level
# of the 21 homology classes: sum pi over a mouse class and rank the human classes by routed mass.
def query(mouse_class, top=4):
    i = [k for k, v in GARIN_NAMES.items() if v == mouse_class][0]
    row = Crow[i - 1]
    print(f"\nmouse {mouse_class!r} routes to:")
    for j in np.argsort(row)[::-1][:top]:
        print(f"   {100 * row[j]:5.1f} %   {NAMES[j]}{'   <- homologue' if j == i - 1 else ''}")

query('Motor/premotor')     # cortical seed
query('Striatum')           # subcortical seed
print(f"\nBoth rank their expected homologue first of 21 classes, against {100 / K:.1f} % uniform.")
print("Panel g renders that routed distribution on the human brain rather than summarising it.")

## 5. The volumetric panels (a, b, c, g)

These are glass-brain renderings and need the Allen and MNI reference volumes. Each panel script
computes from the atlases and the coupling and reads nothing from `outputs/logs/`.

The cell below regenerates all four into `manuscript/figures/fig1/`, which takes a few minutes.
Set `RUN = False` to skip.

In [ ]:
RUN = True

PANELS = [
    ('a, b', FIGS / 'fig1' / 'make_fig1_motivation.py'),
    ('c',    FIGS / 'fig1' / 'make_fig1c_anchors.py'),
    ('g',    FIGS / 'fig1' / 'make_fig1c.py'),
]

if RUN:
    for panel, script in PANELS:
        print(f"--- panel {panel}: {script.name}")
        r = subprocess.run([sys.executable, str(script)], cwd=str(ROOT),
                           capture_output=True, text=True)
        print((r.stdout or r.stderr).strip()[-600:])
        if r.returncode != 0:
            print(f"    FAILED (exit {r.returncode})")
else:
    print('skipped; set RUN = True to rebuild the volumetric panels')

## 6. Summary

Each value below was computed in this notebook and compared against the figure caption.

In [ ]:
print(f"coupling                          {pi.shape[0]:,} x {pi.shape[1]:,}  ({prov['pi_file']})")
print(f"median top-target probability     {med:.2f}")
print(f"parcels with top probability >0.5 {frac_sharp * 100:.0f} %")
print(f"mean self-mass on the diagonal    {self_mass:.2f}  ({self_mass / uniform:.1f}x uniform)")
print(f"topographic fidelity              r = {r_obs:.2f}  (null {null.mean():+.2f})")
print()
ok = all([check('median top-target probability', med),
          check('fraction of parcels above 0.5', frac_sharp),
          check('mean self-mass on the diagonal', self_mass),
          check('fold enrichment over uniform', self_mass / uniform),
          check('topographic fidelity r', r_obs)])
print('\nALL CHECKS PASS' if ok else '\nSOME CHECKS FAILED -- text and code have diverged')